# Stream-Based & Adaptive Learning Algorithms

> Based on Gama, J. — *Knowledge Discovery from Data Streams* (2010)

---

## What you will learn

| # | Topic |
|---|-------|
| 1 | Gama's definition of adaptive learning |
| 2 | Recursive (online) mean and standard deviation — **sufficient statistics** |
| 3 | Verified worked example from the PDF dataset (n=60) |
| 4 | ADWIN (Adaptive Windowing) drift detector |
| 5 | Online learning with `SGDClassifier.partial_fit` |
| 6 | Hoeffding Tree (VFDT) concept |
| 7 | Prequential (test-then-train) evaluation |
| 8 | Comparison: batch vs incremental on a drifting stream |

---

### Gama's adaptive learning — formal definition

Let $T_t = \{\vec{x}_i, y_i : y = f(\vec{x})\}$ be the set of examples available at time $t \in \{1, 2, 3, \ldots, i\}$.

A learning algorithm is **adaptive** if from the sequence of examples
$\{\ldots, E_{j-1}, E_j, \ldots\}$
it produces a sequence of hypotheses
$\{\ldots, H_{j-1}, H_j, \ldots\}$
where each hypothesis $H_i$ only depends on the **previous hypothesis $H_{i-1}$** and the **current example $E_i$**.

An adaptive algorithm requires two operators:
- **Increment** — example $E_k$ is *incorporated* into the decision model.
- **Decrement** — example $E_k$ is *forgotten* from the decision model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Imports OK')

---
## 1  Sufficient Statistics on Streams

Gama's key insight: to compute **mean** and **standard deviation** incrementally on a stream, we only ever need to maintain three counters:

| Statistic | Symbol | Update rule |
|-----------|--------|-------------|
| Count | $i$ | $i \leftarrow i + 1$ |
| Sum | $S = \sum_{k=1}^i x_{k,j}$ | $S \leftarrow S + x_{i,j}$ |
| Sum of squares | $SS = \sum_{k=1}^i x_{k,j}^2$ | $SS \leftarrow SS + x_{i,j}^2$ |

Then at any point:

$$\bar{x}_{i,j} = \frac{S}{i}$$

$$\sigma_{i,j} = \sqrt{\frac{SS - \frac{S^2}{i}}{i-1}}$$

### Recursive (online) mean:

$$\bar{x}_{i,j} = \frac{(i-1) \cdot \bar{x}_{i-1,j} + x_{i,j}}{i}$$

### 1.1  Worked example — dataset from the lecture (n = 60)

Dataset: `4,3,2,5,4,6,3,7,4,1,4,0,6,4,3,5,2,3,5,1,4,4,9,5,4,3,3,5,2,4,3,6,5,2,6,2,4,5,5,1,5,4,4,2,7,1,3,3,4,7,3,4,4,6,6,3,3,2,6,1`

In [ ]:
data = np.array([4,3,2,5,4,6,3,7,4,1,4,0,6,4,3,5,2,3,5,1,
                 4,4,9,5,4,3,3,5,2,4,3,6,5,2,6,2,4,5,5,1,
                 5,4,4,2,7,1,3,3,4,7,3,4,4,6,6,3,3,2,6,1])

n = len(data)
print(f'n = {n}')

# --- Batch mean (formula from PDF) ---
batch_mean = data.sum() / n
print(f'\nBatch mean  = {data.sum()} / {n} = {batch_mean:.4f}')

# --- Sufficient statistics ---
S  = data.sum()
SS = (data**2).sum()
batch_std = np.sqrt((SS - S**2 / n) / (n - 1))
print(f'Sum        S  = {S}')
print(f'Sum-sq     SS = {SS}')
print(f'Batch std  σ  = sqrt(({SS} - {S}²/{n}) / {n-1}) = {batch_std:.4f}')

# NumPy sanity check
print(f'\nnp.mean = {data.mean():.4f}  (expected ≈ 3.8667)')
print(f'np.std  = {data.std(ddof=1):.4f}  (expected ≈ 1.789)')

### 1.2  Recursive mean — online update

In [ ]:
running_mean = 0.0
means = []

for i, x in enumerate(data, start=1):
    # Gama recursive formula: x̄_i = ((i-1)*x̄_{i-1} + x_i) / i
    running_mean = ((i - 1) * running_mean + x) / i
    means.append(running_mean)

print(f'Final recursive mean = {running_mean:.4f}')
print(f'Matches batch mean   = {np.isclose(running_mean, batch_mean)}')

plt.figure(figsize=(10, 4))
plt.plot(range(1, n+1), means, lw=2, label='Recursive mean')
plt.axhline(batch_mean, color='red', ls='--', label=f'Batch mean = {batch_mean:.4f}')
plt.xlabel('Samples seen (i)')
plt.ylabel('Mean estimate')
plt.title('Online Convergence of Recursive Mean')
plt.legend()
plt.tight_layout()
plt.show()

### 1.3  Online standard deviation — Welford's algorithm

Welford (1962) maintains the same three sufficient statistics, numerically stable:

In [ ]:
class OnlineStats:
    """Welford online algorithm for mean and variance."""
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self._M2 = 0.0   # sum of squared deviations

    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self._M2 += delta * delta2

    @property
    def variance(self):
        return self._M2 / (self.n - 1) if self.n >= 2 else 0.0

    @property
    def std(self):
        return np.sqrt(self.variance)


stats = OnlineStats()
online_means, online_stds = [], []

for x in data:
    stats.update(x)
    online_means.append(stats.mean)
    online_stds.append(stats.std)

print(f'Online mean = {stats.mean:.4f}  (batch = {batch_mean:.4f})')
print(f'Online std  = {stats.std:.4f}  (batch = {batch_std:.4f})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(online_means, lw=2)
axes[0].axhline(batch_mean, color='red', ls='--', label='Batch')
axes[0].set_title('Online Mean')
axes[0].legend()

axes[1].plot(online_stds, lw=2, color='orange')
axes[1].axhline(batch_std, color='red', ls='--', label='Batch')
axes[1].set_title('Online Std Dev')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2  ADWIN — Adaptive Windowing Drift Detector

ADWIN (Bifet & Gavalda, 2007) is the standard drift detector for streams.  
It maintains an adaptive window W and detects concept drift by checking whether any two subwindows of W have significantly different means.

We implement a simple ADWIN-style detector using sufficient statistics and the Hoeffding bound:

$$\epsilon_{cut} = \sqrt{\frac{1}{2m^*} \ln \frac{4n}{\delta}}$$

where $m^* = \frac{1}{1/n_0 + 1/n_1}$ is the harmonic mean of the two subwindow sizes.

In [ ]:
class SimpleADWIN:
    """
    Lightweight ADWIN-style drift detector.
    Uses Hoeffding bound to detect significant mean shifts.
    """
    def __init__(self, delta: float = 0.002):
        self.delta = delta
        self.window: list[float] = []
        self.drift_detected = False

    def add_element(self, value: float) -> bool:
        self.window.append(value)
        self.drift_detected = False

        n = len(self.window)
        if n < 4:
            return False

        # Slide the cut point and test each partition
        w = np.array(self.window)
        for cut in range(1, n):
            w0, w1 = w[:cut], w[cut:]
            n0, n1 = len(w0), len(w1)
            m_star = 1.0 / (1.0/n0 + 1.0/n1)
            eps_cut = np.sqrt(np.log(4 * n / self.delta) / (2 * m_star))

            if abs(w0.mean() - w1.mean()) >= eps_cut:
                # Drift: drop the older subwindow
                self.window = list(w1)
                self.drift_detected = True
                return True
        return False


# ---- Simulate a stream with an abrupt drift at t=500 ----
T = 1000
drift_at = 500
stream = np.concatenate([
    np.random.normal(loc=0.0, scale=1.0, size=drift_at),
    np.random.normal(loc=3.0, scale=1.0, size=T - drift_at),
])

detector = SimpleADWIN(delta=0.002)
drift_points = []

for t, val in enumerate(stream):
    drifted = detector.add_element(val)
    if drifted:
        drift_points.append(t)

print(f'Drift detected at time-steps: {drift_points[:5]}  (true drift at t={drift_at})')

plt.figure(figsize=(12, 4))
plt.plot(stream, alpha=0.5, lw=0.8, label='Stream')
for dp in drift_points:
    plt.axvline(dp, color='red', alpha=0.4, lw=0.8)
plt.axvline(drift_at, color='black', ls='--', lw=2, label=f'True drift at t={drift_at}')
red_patch = mpatches.Patch(color='red', alpha=0.5, label='Detected drift')
plt.legend(handles=[plt.Line2D([0],[0],color='black',ls='--',lw=2,label=f'True drift (t={drift_at})'),
                    red_patch])
plt.title('ADWIN Drift Detection on Simulated Stream')
plt.xlabel('Time step')
plt.ylabel('Value')
plt.tight_layout()
plt.show()

---
## 3  Online Learning with `partial_fit` (Increment operator)

scikit-learn's `SGDClassifier` implements exactly Gama's adaptive algorithm:
- **Increment**: `partial_fit(X_i, y_i)` — incorporate the new example.
- The model only stores the current hypothesis (weights), not past data.

We test on a **drifting binary stream** and compare:
- **Batch** model trained once on initial data
- **Incremental** model updated on each new chunk

In [ ]:
from sklearn.datasets import make_classification

def make_drifting_stream(n_initial=500, n_drift=1000, n_features=10, random_state=42):
    """Generates initial data + a concept-drifted stream (flipped class boundaries)."""
    rng = np.random.default_rng(random_state)

    X0, y0 = make_classification(n_samples=n_initial, n_features=n_features,
                                  random_state=random_state)
    # Drift: flip sign of half the features
    X_drift, y_drift = make_classification(n_samples=n_drift, n_features=n_features,
                                            random_state=random_state + 1)
    X_drift[:, :n_features//2] *= -1

    return X0, y0, X_drift, y_drift

X_init, y_init, X_stream, y_stream = make_drifting_stream()

scaler = StandardScaler().fit(X_init)
X_init_s  = scaler.transform(X_init)
X_stream_s = scaler.transform(X_stream)

print(f'Initial: {X_init.shape}  |  Stream (drifted): {X_stream.shape}')

In [ ]:
CHUNK = 50      # process stream in chunks of 50 (mini-batch online learning)
CLASSES = [0, 1]

# --- Batch model: trained once, never updated ---
batch_clf = SGDClassifier(loss='log_loss', random_state=RANDOM_STATE)
batch_clf.fit(X_init_s, y_init)

# --- Incremental model: partial_fit on every new chunk ---
incr_clf = SGDClassifier(loss='log_loss', random_state=RANDOM_STATE)
incr_clf.fit(X_init_s, y_init)  # warm start with initial data

batch_accs, incr_accs, chunk_ids = [], [], []

for start in range(0, len(X_stream_s) - CHUNK, CHUNK):
    X_chunk = X_stream_s[start : start + CHUNK]
    y_chunk = y_stream[start : start + CHUNK]

    # Prequential evaluation: test BEFORE training (test-then-train)
    batch_accs.append(accuracy_score(y_chunk, batch_clf.predict(X_chunk)))
    incr_accs.append(accuracy_score(y_chunk, incr_clf.predict(X_chunk)))

    # Increment: update the incremental model
    incr_clf.partial_fit(X_chunk, y_chunk, classes=CLASSES)

    chunk_ids.append(start + CHUNK)

plt.figure(figsize=(12, 4))
plt.plot(chunk_ids, batch_accs, label='Batch (static)', lw=2)
plt.plot(chunk_ids, incr_accs, label='Incremental (adaptive)', lw=2)
plt.xlabel('Samples seen')
plt.ylabel('Prequential Accuracy')
plt.title('Batch vs Adaptive Learning on Drifting Stream')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Mean accuracy — Batch: {np.mean(batch_accs):.3f}  |  Incremental: {np.mean(incr_accs):.3f}')

---
## 4  Hoeffding Tree (VFDT) — Concept

The Hoeffding Tree (Very Fast Decision Tree) is the canonical adaptive decision tree for data streams.

**Core idea:** At each leaf, accumulate sufficient statistics and only split when we are statistically confident the best split at the current node is better than the second best, using the **Hoeffding bound**:

$$\epsilon = \sqrt{\frac{R^2 \ln(1/\delta)}{2n}}$$

Split occurs when: $\Delta G > \epsilon$, where $\Delta G = G(x_a) - G(x_b)$ is the gain difference between top-2 split candidates.

- $R$ — range of the split metric (e.g. for Gini, $R=1$)
- $\delta$ — confidence parameter
- $n$ — number of examples seen at that leaf

In [ ]:
def hoeffding_bound(R: float, delta: float, n: int) -> float:
    """Hoeffding bound — minimum gain difference to trigger a split."""
    if n == 0:
        return R
    return np.sqrt((R**2 * np.log(1.0 / delta)) / (2 * n))


# Show how the bound shrinks as more examples are seen
n_range = np.arange(1, 5001)
eps_001 = [hoeffding_bound(1.0, 0.01, n) for n in n_range]
eps_0001 = [hoeffding_bound(1.0, 0.001, n) for n in n_range]

plt.figure(figsize=(9, 4))
plt.plot(n_range, eps_001, label='δ = 0.01')
plt.plot(n_range, eps_0001, label='δ = 0.001')
plt.xlabel('Examples seen at leaf (n)')
plt.ylabel('Hoeffding bound ε')
plt.title('Hoeffding Bound vs Examples Seen (R=1)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'At n=100,  ε(δ=0.01) = {hoeffding_bound(1,0.01,100):.4f}')
print(f'At n=1000, ε(δ=0.01) = {hoeffding_bound(1,0.01,1000):.4f}')
print(f'At n=5000, ε(δ=0.01) = {hoeffding_bound(1,0.01,5000):.4f}')

---
## 5  Prequential Evaluation

Standard batch evaluation (train/test split) is **not valid** for streams.  
**Prequential** (progressive validation): for each example, **test first, then train**.

This gives an unbiased, online estimate of accuracy at every time step.

In [ ]:
def prequential_eval(X_stream, y_stream, model, chunk_size=50, fade_factor=0.99):
    """
    Prequential (fading window) evaluation.
    Uses an exponential fading factor to weight recent accuracy more.
    """
    fading_acc = []
    numerator, denominator = 0.0, 0.0

    for start in range(0, len(X_stream) - chunk_size, chunk_size):
        Xc = X_stream[start:start+chunk_size]
        yc = y_stream[start:start+chunk_size]

        preds = model.predict(Xc)
        correct = (preds == yc).sum()

        numerator = fade_factor * numerator + correct
        denominator = fade_factor * denominator + chunk_size
        fading_acc.append(numerator / denominator)

        model.partial_fit(Xc, yc, classes=[0, 1])

    return fading_acc


model_prequential = SGDClassifier(loss='log_loss', random_state=RANDOM_STATE)
model_prequential.fit(X_init_s, y_init)

fading_acc = prequential_eval(X_stream_s, y_stream, model_prequential)

plt.figure(figsize=(10, 4))
plt.plot(fading_acc, lw=2)
plt.xlabel('Chunk index')
plt.ylabel('Fading Accuracy')
plt.title('Prequential Evaluation (fading window, α=0.99)')
plt.tight_layout()
plt.show()

---
## 6  Summary

| Concept | Key formula / idea |
|---|---|
| Adaptive algorithm (Gama) | $H_i$ depends only on $H_{i-1}$ and $E_i$ |
| Recursive mean | $\bar{x}_i = \frac{(i-1)\bar{x}_{i-1} + x_i}{i}$ |
| Online std (Welford) | Maintain count, sum, sum-of-sq — O(1) per sample |
| ADWIN | Hoeffding-bound windowed drift detector |
| Increment operator | `partial_fit(X_i, y_i)` |
| Hoeffding Tree split criterion | Split when $\Delta G > \sqrt{R^2 \ln(1/\delta) / 2n}$ |
| Prequential evaluation | Test-then-train; valid for streaming data |

### Next steps
- Install `river` (`pip install river`) for production-grade Hoeffding Trees, ADWIN, and 50+ stream learners
- Apply these concepts to the time-series notebook (116) for online demand forecasting
- Explore **forgetting mechanisms**: sliding window, fading memory, concept drift recovery